In [17]:
# =============================================================================
# Associações mistas (quantitativas + qualitativas) com detecção de não linearidade
# Dependências mínimas: pandas, numpy, scipy, statsmodels, patsy
# Saídas:
#   - Tabela "longa" com uma linha por (par de variáveis × método)
#   - Tipos detectados por coluna (numérica, binária, categórica, ordinal)
#
# Medidas/respectivos pares:
#   - numérico–numérico: Pearson (linear), Spearman (monotônica),
#                         Distance Correlation (qualquer dependência),
#                         teste spline vs linear (p-valor de não linearidade)
#   - numérico–binário : Correlação ponto-biserial
#   - numérico–nominal : Eta (correlation ratio) + p-valor por permutação
#   - ordinal–ordinal  : Kendall tau-b
#   - categórico–categórico (inclui binário): Cramér's V (χ²), com correção de viés
#
# Observações:
#   * A tabela inclui p-values e q-values (FDR BH). Use `significant_(q<0.05)` para filtrar descobertas.
#   * Distance Correlation é O(n^2); use `max_n_dcor` para amostrar em bases muito grandes.
#   * O teste spline vs linear informa apenas p-valor (sem tamanho de efeito).
# =============================================================================

# Funcoes

In [18]:
import numpy as np
import pandas as pd
from itertools import combinations
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from patsy import bs  # base splines p/ teste de não linearidade

In [19]:
# --------------------------- Utilitários ---------------------------

def _pairwise_xy(x, y):
    """
    Alinha duas séries (ou arrays), removendo NAs em par (listwise) e
    retorna (x_limpo, y_limpo, n).
    """
    pair = pd.DataFrame({"x": x, "y": y}).dropna()
    return pair["x"].to_numpy(), pair["y"].to_numpy(), len(pair)


def _var_type(s: pd.Series):
    """
    Detecta tipo simples de variável para escolher a medida de associação:
    - 'numeric'   : numérica contínua
    - 'binary'    : 0/1, bool, ou numérica com exatamente 2 valores distintos
    - 'categorical': nominal (string/objeto/categórica não ordenada)
    - 'ordinal'   : categórica ordenada (dtype 'category' com ordered=True)
    - 'unknown'
    """
    s = s.dropna()
    if s.empty:
        return "unknown"
    if pd.api.types.is_bool_dtype(s):
        return "binary"
    if pd.api.types.is_numeric_dtype(s):
        return "binary" if s.nunique(dropna=True) == 2 else "numeric"
    if pd.api.types.is_categorical_dtype(s):
        return "ordinal" if getattr(s.dtype, "ordered", False) else "categorical"
    # objetos/strings tratamos como categórico nominal
    return "categorical"


def _ordinal_codes(s: pd.Series):
    """
    Converte ordinal/categórica em códigos inteiros (0,1,2,...).
    Mantém numérico como float.
    Útil para aplicar Spearman/Kendall em ordinais.
    """
    if pd.api.types.is_categorical_dtype(s):
        return s.cat.codes.to_numpy().astype(float)
    if pd.api.types.is_numeric_dtype(s):
        return s.to_numpy().astype(float)
    # fallback para objetos/strings
    return pd.factorize(s, sort=True)[0].astype(float)


def _stat_p(res):
    """
    Compatibilidade com SciPy: algumas funções retornam objetos com
    atributos .statistic/.pvalue; outras retornam tuplas (stat, p).
    """
    if hasattr(res, "statistic") and hasattr(res, "pvalue"):
        return float(res.statistic), float(res.pvalue)
    # namedtuple/tupla
    try:
        stat, p = res
        return float(stat), float(p)
    except Exception:
        return np.nan, np.nan

In [20]:
# -------------------- Medidas de associação ----------------------

def pearson_effect(x, y):
    x, y, n = _pairwise_xy(x, y)
    if n < 5:
        return {"method": "pearson_r", "effect": np.nan, "p_value": np.nan, "n": n}
    r, p = _stat_p(stats.pearsonr(x, y))
    return {"method": "pearson_r", "effect": r, "p_value": p, "n": int(n)}


def spearman_effect(x, y):
    x, y, n = _pairwise_xy(x, y)
    if n < 5:
        return {"method": "spearman_rho", "effect": np.nan, "p_value": np.nan, "n": n}
    r, p = _stat_p(stats.spearmanr(x, y))
    return {"method": "spearman_rho", "effect": r, "p_value": p, "n": int(n)}


def kendall_tau(x, y):
    x, y, n = _pairwise_xy(x, y)
    if n < 5:
        return {"method": "kendall_tau_b", "effect": np.nan, "p_value": np.nan, "n": n}
    r, p = _stat_p(stats.kendalltau(x, y))
    return {"method": "kendall_tau_b", "effect": r, "p_value": p, "n": int(n)}


def pointbiserial_effect(x_num, y_bin):
    """
    Correlação ponto-biserial: equivalente ao Pearson entre (x_num) e (y_bin in {0,1}).
    """
    x, y, n = _pairwise_xy(pd.Series(x_num).astype(float), pd.Series(y_bin).astype(int))
    if n < 5 or (np.all(y == y[0])):  # precisa de variância nos grupos
        return {"method": "point_biserial", "effect": np.nan, "p_value": np.nan, "n": n}
    r, p = _stat_p(stats.pointbiserialr(x, y))
    return {"method": "point_biserial", "effect": r, "p_value": p, "n": int(n)}


def cramers_v_chi2(x_cat, y_cat):
    """
    Associação categórico–categórico via χ² + Cramér's V com correção de viés.
    """
    pair = pd.DataFrame({"x": x_cat, "y": y_cat}).dropna()
    if pair.empty:
        return {"method": "cramers_v", "effect": np.nan, "p_value": np.nan, "n": 0}
    tbl = pd.crosstab(pair["x"], pair["y"])
    if tbl.shape[0] < 2 or tbl.shape[1] < 2:
        return {"method": "cramers_v", "effect": np.nan, "p_value": np.nan, "n": int(tbl.values.sum())}
    chi2, p, dof, exp = stats.chi2_contingency(tbl, correction=False)
    n = tbl.values.sum()
    r, k = tbl.shape
    # Correção de viés (Bergsma)
    phi2 = chi2 / n
    phi2corr = max(0.0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    denom = min((kcorr - 1), (rcorr - 1))
    V = 0.0 if denom <= 0 else np.sqrt(phi2corr / denom)
    return {"method": "cramers_v", "effect": float(V), "p_value": float(p), "n": int(n)}


def correlation_ratio_eta(y_num, groups, n_perm=2000, random_state=0):
    """
    Relação de correlação (eta) para numérico–nominal:
      eta = sqrt(SS_between / SS_total).
    p-valor via permutação mantendo os tamanhos de grupo.
    """
    df = pd.DataFrame({"y": y_num, "g": groups}).dropna()
    n = len(df)
    if n < 5 or df["g"].nunique() < 2:
        return {"method": "correlation_ratio_eta", "effect": np.nan, "p_value": np.nan, "n": n}

    y = df["y"].to_numpy().astype(float)
    g_codes, g_uniques = pd.factorize(df["g"], sort=True)
    overall = y.mean()
    ss_total = ((y - overall) ** 2).sum()

    # Soma de quadrados entre grupos (SS_between)
    ss_between = 0.0
    for code in range(len(g_uniques)):
        yi = y[g_codes == code]
        ss_between += len(yi) * (yi.mean() - overall) ** 2
    eta = 0.0 if ss_total == 0 else float(np.sqrt(ss_between / ss_total))

    # p-valor por permutação
    if n_perm is None or n < 5:
        return {"method": "correlation_ratio_eta", "effect": eta, "p_value": np.nan, "n": n}
    rng = np.random.default_rng(random_state)
    count = 0
    for _ in range(int(n_perm)):
        yp = rng.permutation(y)
        overall_p = yp.mean()
        ss_b_p = 0.0
        for code in range(len(g_uniques)):
            yi = yp[g_codes == code]
            ss_b_p += len(yi) * (yi.mean() - overall_p) ** 2
        eta_p = 0.0 if ss_total == 0 else np.sqrt(ss_b_p / ss_total)
        count += (eta_p >= eta)
    pval = (count + 1) / (int(n_perm) + 1)
    return {"method": "correlation_ratio_eta", "effect": eta, "p_value": float(pval), "n": n}

In [21]:
# -------------------- Detecção de não linearidade -----------------------

def _double_center(D):
    """
    Centralização dupla de uma matriz de distâncias (necessária p/ distância-correlação).
    """
    r = D.mean(axis=1, keepdims=True)
    c = D.mean(axis=0, keepdims=True)
    m = D.mean()
    return D - r - c + m


def distance_correlation_test(x, y, n_perm=500, random_state=0, max_n=None):
    """
    Distance Correlation (dCor) com p-valor por permutação.
    Detecta QUALQUER dependência (linear ou não). Complexidade O(n^2).
    Parâmetros:
      - n_perm : nº de permutações (aumente para p-values mais estáveis)
      - max_n  : se não for None e n > max_n, faz amostragem aleatória para acelerar
    Retorna: (dCor, p_value, n_usado)
    """
    x = pd.Series(x).dropna()
    y = pd.Series(y).dropna()
    pair = pd.concat([x, y], axis=1).dropna()
    if pair.shape[0] < 5:
        return np.nan, np.nan, pair.shape[0]

    # Amostragem opcional para acelerar em bases muito grandes
    if max_n is not None and pair.shape[0] > max_n:
        rng = np.random.default_rng(random_state)
        idx = rng.choice(pair.index.to_numpy(), size=max_n, replace=False)
        pair = pair.loc[idx]

    xv = pair.iloc[:, 0].to_numpy().astype(float)
    yv = pair.iloc[:, 1].to_numpy().astype(float)
    n = len(xv)

    # Distâncias euclidianas 1D e centralização dupla
    X = xv.reshape(-1, 1)
    Y = yv.reshape(-1, 1)
    Dx = np.abs(X - X.T)
    Dy = np.abs(Y - Y.T)
    A = _double_center(Dx)
    B = _double_center(Dy)

    dcov2 = (A * B).mean()
    dvarx = (A * A).mean()
    dvary = (B * B).mean()
    dcor = 0.0 if dvarx <= 0 or dvary <= 0 else float(np.sqrt(dcov2 / np.sqrt(dvarx * dvary)))

    # Permutação do vetor Y
    rng = np.random.default_rng(random_state)
    count = 0
    for _ in range(int(n_perm)):
        yp = rng.permutation(Y)
        Dyp = np.abs(yp - yp.T)
        Bp = _double_center(Dyp)
        dcov2_p = (A * Bp).mean()
        dvary_p = (Bp * Bp).mean()
        dcor_p = 0.0 if dvarx <= 0 or dvary_p <= 0 else np.sqrt(dcov2_p / np.sqrt(dvarx * dvary_p))
        count += (dcor_p >= dcor)
    pval = (count + 1) / (int(n_perm) + 1)
    return dcor, float(pval), int(n)


def spline_nonlin_pvalue(x, y, df_spline=5):
    """
    Teste de não linearidade: compara modelos aninhados
      H0: y ~ x (linear)    vs    H1: y ~ bs(x, df=df_spline) (curva)
    Retorna p-valor da ANOVA entre os modelos (menor => curva melhora significativamente).
    """
    pair = pd.DataFrame({"x": x, "y": y}).dropna()
    n = len(pair)
    # Precisamos de variação suficiente em x e tamanho de amostra razoável
    if n < 10 or pair["x"].nunique() < 4:
        return np.nan, n
    lin = smf.ols("y ~ x", data=pair).fit()
    spl = smf.ols(f"y ~ bs(x, df={int(df_spline)})", data=pair).fit()
    anova = sm.stats.anova_lm(lin, spl)
    p = np.nan
    if "Pr(>F)" in anova.columns and len(anova) > 1:
        p = float(anova["Pr(>F)"].iloc[1])
    return p, n

In [22]:
# ------------------------- Função principal --------------------------

def mixed_associations(
    df: pd.DataFrame,
    *,
    add_spearman: bool = True,
    add_distance_corr: bool = True,
    n_perm_dcor: int = 500,
    max_n_dcor: int | None = None,
    add_spline_test: bool = True,
    spline_df: int = 5,
    n_perm_eta: int = 2000,
    fdr_alpha: float = 0.05,
    random_state: int = 0
):
    """
    Varre todos os pares de colunas do DataFrame e calcula a medida de associação adequada,
    adicionando (para numérico–numérico):
      - Distance Correlation (qualquer dependência)
      - Teste spline vs linear (p-valor de não linearidade)

    Parâmetros principais:
      - add_spearman        : inclui Spearman para numérico–numérico
      - add_distance_corr   : inclui Distance Correlation (n_perm_dcor, max_n_dcor)
      - add_spline_test     : inclui teste spline vs linear (spline_df)
      - n_perm_eta          : nº de permutações no eta (numérico–nominal)
      - fdr_alpha           : nível para BH–FDR
      - random_state        : reprodutibilidade (permutação/amostragem)

    Retorna:
      - results_df (pd.DataFrame): colunas [x, y, type_pair, method, effect, p_value, q_value, significant_(q<0.05), n]
      - detected_types (pd.Series): mapeamento coluna -> tipo detectado
    """
    # Seleciona colunas com pelo menos 2 observações não nulas
    cols = [c for c in df.columns if df[c].notna().sum() > 1]
    types = {c: _var_type(df[c]) for c in cols}
    rows = []

    for a, b in combinations(cols, 2):
        ta, tb = types[a], types[b]
        xa, xb = df[a], df[b]

        # Amostra pareada (listwise) para este par
        pair = pd.DataFrame({a: xa, b: xb}).dropna()
        if len(pair) < 5:
            continue

        # --------------------- numérico–numérico ---------------------
        if ta == "numeric" and tb == "numeric":
            rows.append({"x": a, "y": b, "type_pair": "numeric-numeric", **pearson_effect(xa, xb)})

            if add_spearman:
                rows.append({"x": a, "y": b, "type_pair": "numeric-numeric", **spearman_effect(xa, xb)})

            if add_distance_corr:
                dcor, p, n_used = distance_correlation_test(
                    xa, xb, n_perm=n_perm_dcor, random_state=random_state, max_n=max_n_dcor
                )
                rows.append({
                    "x": a, "y": b, "type_pair": "numeric-numeric",
                    "method": "distance_correlation", "effect": dcor, "p_value": p, "n": n_used
                })

            if add_spline_test:
                p_nonlin, n_used = spline_nonlin_pvalue(xa, xb, df_spline=spline_df)
                rows.append({
                    "x": a, "y": b, "type_pair": "numeric-numeric",
                    "method": f"spline_vs_linear(df={int(spline_df)})",
                    "effect": np.nan, "p_value": p_nonlin, "n": n_used
                })

        # --------------------- numérico–binário ----------------------
        elif (ta == "numeric" and tb == "binary") or (tb == "numeric" and ta == "binary"):
            x_num = xa if ta == "numeric" else xb
            y_bin = xb if tb == "binary" else xa
            rows.append({"x": a, "y": b, "type_pair": f"{ta}-{tb}", **pointbiserial_effect(x_num, y_bin)})

        # --------------------- numérico–categórico (nominal) --------
        elif (ta == "numeric" and tb == "categorical") or (tb == "numeric" and ta == "categorical"):
            y_num = xa if ta == "numeric" else xb
            g = xb if tb == "categorical" else xa
            rows.append({"x": a, "y": b, "type_pair": f"{ta}-{tb}",
                        **correlation_ratio_eta(y_num, g, n_perm=n_perm_eta, random_state=random_state)})

        # --------------------- numérico–ordinal ----------------------
        elif (ta == "numeric" and tb == "ordinal") or (tb == "numeric" and ta == "ordinal"):
            xo = _ordinal_codes(xa) if ta == "ordinal" else xa.to_numpy().astype(float)
            yo = _ordinal_codes(xb) if tb == "ordinal" else xb.to_numpy().astype(float)
            r, p = _stat_p(stats.spearmanr(*_pairwise_xy(xo, yo)[:2]))
            n_used = _pairwise_xy(xo, yo)[2]
            rows.append({
                "x": a, "y": b, "type_pair": f"{ta}-{tb}",
                "method": "spearman_rho", "effect": float(r), "p_value": float(p), "n": int(n_used)
            })

        # --------------------- ordinal–ordinal -----------------------
        elif ta == "ordinal" and tb == "ordinal":
            xo = _ordinal_codes(xa)
            yo = _ordinal_codes(xb)
            rows.append({"x": a, "y": b, "type_pair": "ordinal-ordinal", **kendall_tau(xo, yo)})

        # -------- categórico–categórico (inclui binário–categórico) -
        elif (ta in ["categorical", "binary"]) and (tb in ["categorical", "binary"]):
            rows.append({"x": a, "y": b, "type_pair": f"{ta}-{tb}",
                        **cramers_v_chi2(xa.astype("category"), xb.astype("category"))})

        # --------------------- fallback conservador ------------------
        else:
            rows.append({"x": a, "y": b, "type_pair": f"{ta}-{tb} (fallback)",
                        **cramers_v_chi2(xa.astype("category"), xb.astype("category"))})

    out = pd.DataFrame(rows)
    if out.empty:
        return out, pd.Series(types, name="detected_type")

    # ---------------------- Ajuste p/ múltiplos testes (BH–FDR) ----------------------
    valid = out["p_value"].notna()
    qvals = np.full(len(out), np.nan, dtype=float)
    if valid.any():
        _, qv, _, _ = multipletests(out.loc[valid, "p_value"], alpha=fdr_alpha, method="fdr_bh")
        qvals[np.where(valid)[0]] = qv
    out["q_value"] = qvals
    out["significant_(q<0.05)"] = out["q_value"] < 0.05

    # Ordenação amigável: significantes primeiro, depois por q, p, |efeito|
    out["abs_effect"] = out["effect"].abs() if "effect" in out else np.nan
    out = out.sort_values(
        by=["significant_(q<0.05)", "q_value", "p_value", "abs_effect"],
        ascending=[False, True, True, False]
    ).drop(columns=["abs_effect"])

    detected_types = pd.Series(types, name="detected_type")
    return out.reset_index(drop=True), detected_types


# Aplicacoes

In [23]:
df = pd.read_excel("Base_teste_2026.xlsx")

In [24]:
df.head()

,Y,X1,X2,X3,X4,X5
0,164,180,0,D,328,3
1,96,157,0,A,228,2
2,29,77,0,A,30,1
3,220,234,1,D,7,1
4,21,73,1,E,392,3


In [26]:
results, tipos = mixed_associations(
    df,
    add_spearman=True,
    add_distance_corr=True,
    n_perm_dcor=200,     # menor para rodar rápido no início
    max_n_dcor=1000,     # amostra máx. para acelerar o dCor
    add_spline_test=True,
    spline_df=5,
    n_perm_eta=500,      # menor para teste inicial
    fdr_alpha=0.05,
    random_state=42
)

C:\Users\gcabr\AppData\Local\Temp\ipykernel_16228\1427015929.py:28: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(s):
C:\Users\gcabr\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\stats\anova.py:374: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps upd

In [27]:
results

,x,y,type_pair,method,effect,p_value,n,q_value,significant_(q<0.05)
0,X4,X5,numeric-numeric,spearman_rho,0.939840,1.908361e-234,500,6.297593e-233,True
1,X4,X5,numeric-numeric,pearson_r,0.939024,4.927735e-233,500,8.130762e-232,True
2,Y,X1,numeric-numeric,spearman_rho,0.882048,6.999056e-165,500,7.698961e-164,True
3,Y,X1,numeric-numeric,pearson_r,0.875553,1.867429e-159,500,1.540629e-158,True
4,X4,X5,numeric-numeric,spline_vs_linear(df=5),NaN,9.960492e-41,500,6.573925e-40,True
5,X2,X3,binary-categorical,cramers_v,0.486716,1.795032e-25,500,9.872676e-25,True
6,X1,X2,numeric-binary,point_biserial,0.175874,7.695850e-05,500,3.628043e-04,True
7,Y,X2,numeric-binary,point_biserial,0.162752,2.576574e-04,500,1.062837e-03,True
8,X4,X5,numeric-numeric,distance_correlation,0.934062,4.975124e-03,500,1.641791e-02,True
9,Y,X1,numeric-numeric,distance_correlation,0.864352,4.975124e-03,500,1.641791e-02,True


In [ ]:
,
